# FLAG 2027 — re-extract features from raw audio / images

**Why this notebook exists.** Our current system (30.90, top 4) uses the 192-d voice embeddings the
organisers ship. The FAME 2026 winner used the *same* ECAPA-TDNN but took the **6144-d layer before
the output**, plus a richer face encoder. Every internal symptom says we are at the ceiling of the
provided features: a 28-config architecture sweep spread only 0.5 EER, augmentation did not help,
and a closed-form linear CCA already reached 35.79.

**Rules.** The evaluation plan ships the data *"Alongside the audios (.wav) and images (.jpg)"* and
*"provided alongside pre-extracted features"*, and states *"A pretrained encoder for faces or voices
is allowed."* Re-extracting with a frozen pretrained encoder is therefore on solid ground. (Training
on an **external** face-voice corpus such as VoxCeleb2 is a separate question — ask the organisers.)

**Discipline.** This notebook changes exactly one thing at a time:

| run | face | voice | verdict it gives |
|---|---|---|---|
| A (control) | provided 4096-d | provided 192-d | our current 30.90 system |
| B | provided 4096-d | **ECAPA 6144-d** | is the 192-d voice the bottleneck? |
| C | provided 4096-d | ECAPA 192-d (ours) | does our extraction reproduce the organisers' CSV? |
| D | **ArcFace 512-d** | provided 192-d | is VGGFace fc7 the bottleneck? |
| E | ArcFace 512-d | ECAPA 6144-d | both |

**Settings:** GPU T4 x2, **Internet ON** (needed to download the pretrained encoders).

**Input — one dataset is enough:** upload `Input/` (or the two zips). Kaggle may auto-extract them;
the notebook handles either form. The organisers' feature CSVs ship inside `train_set/features/` and
`dev_set/*/features/`, so the control run A and the encoder cross-check are covered by the same upload.


**Important fixes in this version.**
- Submission polarity follows the official FLAG rule: **higher score = higher confidence of a match**.
- ZIP extraction and later feature loading now use the same `WORK` root.
- PCA cache keys are feature-aware, so the 4096-d control PCA cannot leak into the 512-d ArcFace run.
- Local model selection uses the mean of standard and gender-constrained internal EER as a closer proxy to the official multi-cell average.


In [6]:
# Writes flag_lib.py (the evaluation code from the sweep notebook) next to this notebook.
import base64
_SRC = '''
...
'''
open('flag_lib.py', 'wb').write(base64.b64decode(''.join(_SRC.split())))
print('flag_lib.py written')


flag_lib.py written


## 1. Unpack the official raw media


In [7]:
!pip -q install speechbrain==1.0.2 2>&1 | tail -2
import os, sys, time, zipfile, json
from pathlib import Path
import numpy as np, pandas as pd, torch, torchaudio

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEV, "| GPUs", torch.cuda.device_count())

IN = Path("/kaggle/input")

# Prefer already-extracted raw media. Otherwise extract the attached train/dev archives.
raw_wavs = list(IN.rglob("*.wav"))
if len(raw_wavs) > 10000:
    WORK = IN
    wavs = raw_wavs
    print(f"raw media already extracted by Kaggle: {len(wavs)} wav found, reading in place")
else:
    zips = sorted(IN.rglob("*.zip"))
    preferred = [z for z in zips if z.name.lower() in {"train_set.zip", "dev_set.zip"}]
    zips = preferred or zips
    assert zips, ("No complete raw media and no .zip under /kaggle/input - attach the FLAG raw "
                  "train/dev data (train_set.zip + dev_set.zip, or their extracted form).")
    WORK = Path("/kaggle/working/raw")
    WORK.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for zp in zips:
        with zipfile.ZipFile(zp) as z:
            z.extractall(WORK)
        print("extracted", zp.name, round(time.time() - t0), "s", flush=True)
    wavs = list(WORK.rglob("*.wav"))

jpgs = list(WORK.rglob("*.jpg"))
print(f"wav: {len(wavs)} | jpg: {len(jpgs)}")
assert len(wavs) > 10000, f"expected the full FLAG raw release, found only {len(wavs)} wav files"

# IMPORTANT: all later feature/txt discovery must use the same root that actually contains
# the extracted data.  The old notebook searched /kaggle/input even after extracting to WORK.
os.environ["FLAG_DATA"] = str(WORK)

CSVS = sorted(WORK.rglob("train_English_voices.csv"))
if not CSVS and WORK != IN:
    CSVS = sorted(IN.rglob("train_English_voices.csv"))
assert CSVS, ("The organisers' train_English_voices.csv is required for the control run. "
              "Attach the official train/dev release including pre-extracted features.")
print("organiser CSV:", CSVS[0])
print("FLAG_DATA:", os.environ["FLAG_DATA"])


torch 2.10.0+cu128 | device cuda | GPUs 2
raw media already extracted by Kaggle: 11349 wav found, reading in place
wav: 11349 | jpg: 11349
organiser CSV: /kaggle/input/datasets/chnnhnnguyn/input-flag2027-features/train/train_English_voices.csv


In [8]:
# Row order must match the organisers' .txt files exactly - their CSVs are aligned to them,
# so our new features stay directly comparable.
#
# Careful: the same .txt name can appear in several attached datasets (e.g. a features-only upload
# that has no media). Pick the copy whose media files actually resolve next to it.
def resolve_split(txt_name, wav_col, jpg_col, must_contain=None):
    cands = [p for p in WORK.rglob(txt_name) if must_contain is None or must_contain in p.as_posix()]
    assert cands, f"no {txt_name} found (filter={must_contain})"
    tried = []
    for p in sorted(cands, key=lambda q: len(q.as_posix())):
        df = pd.read_csv(p, sep=r"\s+", header=None)
        wav, jpg = df[wav_col].tolist(), df[jpg_col].tolist()
        if (p.parent / wav[0]).exists() and (p.parent / jpg[0]).exists():
            return dict(root=p.parent, wav=wav, jpg=jpg)
        tried.append(str(p.parent))
    raise FileNotFoundError(
        f"{txt_name}: found {len(cands)} copies but none has its media beside it. "
        f"Checked these folders: {tried}. "
        "Attach the RAW dataset (the one containing the .wav and .jpg files).")

SPLITS = {"train": resolve_split("train_English.txt", 2, 3)}
for prot in ["no_gender", "gender"]:
    for lang in ["English", "Bangla"]:
        SPLITS[f"{prot}/{lang}"] = resolve_split(f"{lang}_test.txt", 1, 2, must_contain=f"/{prot}/")

for k, v in SPLITS.items():
    root = v["root"]
    shown = root.relative_to(WORK) if WORK in root.parents or WORK == root else root
    print(f"{k:22s} {len(v['wav']):5d} rows | root={shown}")
print("total files to encode:", sum(len(v["wav"]) for v in SPLITS.values()))


train                   6485 rows | root=datasets/trngsnam/flag-input/train_set/train_set/train_set
no_gender/English       1008 rows | root=datasets/trngsnam/flag-input/dev_set/dev_set/no_gender
no_gender/Bangla        1406 rows | root=datasets/trngsnam/flag-input/dev_set/dev_set/no_gender
gender/English           982 rows | root=datasets/trngsnam/flag-input/dev_set/dev_set/gender
gender/Bangla           1468 rows | root=datasets/trngsnam/flag-input/dev_set/dev_set/gender
total files to encode: 11349


## 2. Voice — ECAPA-TDNN, both the 192-d output and the 6144-d pooling layer


In [9]:
from speechbrain.inference.speaker import EncoderClassifier

ecapa = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                       savedir="/kaggle/working/ecapa", run_opts={"device": str(DEV)})
emb_model = ecapa.mods.embedding_model.eval()

# ECAPA: ... -> attentive statistics pooling (6144-d) -> BN -> conv1d (192-d).
# Hook the pooling output to get the richer representation the FAME 2026 winner used.
_pool = {}
emb_model.asp.register_forward_hook(lambda m, i, o: _pool.__setitem__("x", o.detach()))

@torch.no_grad()
def voice_embed(paths, root, bs=16):
    out192, out6144 = [], []
    for s in range(0, len(paths), bs):
        batch = paths[s:s + bs]
        sigs, lens = [], []
        for p in batch:
            w, sr = torchaudio.load(str(Path(root) / p))
            assert sr == 16000, sr
            sigs.append(w[0]); lens.append(w.shape[1])
        L = max(lens)
        x = torch.zeros(len(sigs), L)
        for i, w in enumerate(sigs):
            x[i, :len(w)] = w
        rel = torch.tensor([l / L for l in lens], dtype=torch.float32)
        feats = ecapa.mods.compute_features(x.to(DEV))
        feats = ecapa.mods.mean_var_norm(feats, rel.to(DEV))
        e = emb_model(feats, rel.to(DEV))              # (B, 1, 192)
        out192.append(e.squeeze(1).cpu().numpy())
        out6144.append(_pool["x"].squeeze(-1).cpu().numpy())
        if s % (bs * 50) == 0:
            print(f"   {s}/{len(paths)}", flush=True)
    return np.concatenate(out192), np.concatenate(out6144)

VOICE_FEATS = {}
for name, sp in SPLITS.items():
    t0 = time.time()
    a, b = voice_embed(sp["wav"], sp["root"])
    VOICE_FEATS[name] = dict(ecapa192=a, ecapa6144=b)
    np.save(f"voice_ecapa192_{name.replace('/', '_')}.npy", a)
    np.save(f"voice_ecapa6144_{name.replace('/', '_')}.npy", b)
    print(f"{name:22s} {a.shape} {b.shape}  ({round(time.time()-t0)}s)", flush=True)


AttributeError: module 'torchaudio' has no attribute 'list_audio_backends'

In [ ]:
# Free sanity check: is the organisers' 192-d CSV this same ECAPA?
# (Only possible where we also have their CSV — i.e. if the feature dataset is attached too.)
feat_csv = CSVS   # located in the setup cell
if feat_csv:
    given = pd.read_csv(feat_csv[0], header=None).iloc[:, :-1].values.astype(np.float32)
    ours = VOICE_FEATS["train"]["ecapa192"]
    gn = given / np.linalg.norm(given, axis=1, keepdims=True)
    on = ours / np.linalg.norm(ours, axis=1, keepdims=True)
    diag = np.sum(gn * on, 1)
    print(f"cosine(ours_192, organisers_192): mean {diag.mean():.4f}  median {np.median(diag):.4f}")
    print("norm ratio (ours/given):", round(float(np.linalg.norm(ours, axis=1).mean() /
                                                  np.linalg.norm(given, axis=1).mean()), 3))
    print("-> ~1.0 means the organisers used this exact encoder and 6144-d really is 'the layer before'.")
else:
    print("organiser feature CSVs not attached; skipping this check")


## 3. Face — ArcFace R100 (buffalo_l)


In [ ]:
!pip -q install insightface onnxruntime-gpu 2>&1 | tail -2
import insightface, cv2
from insightface.app import FaceAnalysis

# ArcFace R100 (buffalo_l). Faces are already cropped to 224x224, so we bypass detection and
# feed the whole crop to the recognition model.
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(224, 224))
rec = app.models["recognition"]

def face_embed(paths, root, bs=32):
    out = []
    for s in range(0, len(paths), bs):
        imgs = []
        for p in paths[s:s + bs]:
            im = cv2.imread(str(Path(root) / p))
            imgs.append(cv2.resize(im, (112, 112)))
        blob = cv2.dnn.blobFromImages(np.array(imgs), 1.0 / 127.5, (112, 112), (127.5, 127.5, 127.5), swapRB=True)
        out.append(rec.session.run(rec.output_names, {rec.input_name: blob})[0])
        if s % (bs * 50) == 0:
            print(f"   {s}/{len(paths)}", flush=True)
    return np.concatenate(out)

FACE_FEATS = {}
for name, sp in SPLITS.items():
    t0 = time.time()
    f = face_embed(sp["jpg"], sp["root"])
    FACE_FEATS[name] = f
    np.save(f"face_arcface_{name.replace('/', '_')}.npy", f)
    print(f"{name:22s} {f.shape}  ({round(time.time()-t0)}s)", flush=True)


## 3.5. Logic preflight — score polarity and EER


In [ ]:
# Fast synthetic check before spending GPU time on the sweep.
# Perfect same-speaker scores must give ~0% EER; inverted scores must be very bad.
os.environ["FLAG_DATA"] = str(WORK)
import importlib, flag_lib
Lcheck = importlib.reload(flag_lib)

_lab = np.array([1, 1, 1, 0, 0, 0])
_good = np.array([0.95, 0.85, 0.75, 0.25, 0.15, 0.05])
_bad = -_good
eer_good = Lcheck.eer_from_scores(_good, _lab)
eer_bad = Lcheck.eer_from_scores(_bad, _lab)
print("EER correct polarity:", round(eer_good, 4), "%")
print("EER inverted polarity:", round(eer_bad, 4), "%")
assert eer_good < 1.0 and eer_bad > 90.0
print("OK: higher score means more likely SAME speaker.")


## 4. The decisive ablation — one variable at a time


In [ ]:
# The decisive comparison. Same pipeline, same loss, same protocol — only the features change.
os.environ["FLAG_DATA"] = str(WORK)
import importlib, flag_lib
L = importlib.reload(flag_lib)
L.DATA = Path(WORK)
L._FOUND.clear()
print("flag_lib DATA:", L.DATA)

Xf_old, Xv_old, spk, gmap, vgrp = L.load_train()      # organiser features, for the control run
ALL = np.arange(len(spk))
SEEDS = [1, 2, 3]
SPL = {s: (lambda tv: (np.where(tv[0])[0], np.where(tv[1])[0]))(L.speaker_split(spk, s)) for s in SEEDS}
TRIALS = {(s, sg): L.build_trials(spk[SPL[s][1]], gmap, seed=s, n_pos=3000, n_neg=3000, same_gender=sg)
          for s in SEEDS for sg in [False, True]}

def evaluate(Xf, Xv, tag, cfg=None, n_seeds=3, cca_weight=0.25):
    """Trains our current recipe on the given feature pair and returns (int_ng, int_g)."""
    cfg = cfg or dict(hid=512, emb=128, epochs=40, drop=0.3, dedup_clip=False)
    out = {"no_gender": [], "gender": []}
    for s in SEEDS:
        tri, vai = SPL[s]
        pr = L.Prep(Xf[tri], Xv[tri], key=f"{tag}_{s}")
        aF, aV = L.Aligner(pr.f(Xf[tri])), L.Aligner(pr.v(Xv[tri]))
        embs = []
        for i in range(n_seeds):
            net, prep, _ = L.train_model(Xf, Xv, spk, gmap, vgrp, tri, cfg, seed=s + 100 * i)
            embs.append(L.embed_one(net, prep, Xf[vai], Xv[vai], alF=aF, alV=aV))
        m = L.RidgeCCA(k=4, reg=1.0, pca_x=min(128, Xf.shape[1])).fit(Xf[tri], Xv[tri])
        cca = (m.transform_x(L.center_to(Xf[vai], Xf[tri].mean(0))),
               m.transform_y(L.center_to(Xv[vai], Xv[tri].mean(0))))
        for sg in [False, True]:
            fi, vj, lab = TRIALS[(s, sg)]
            sc = L.fuse_scores(embs, fi, vj, cca=cca, cca_weight=cca_weight)
            out["gender" if sg else "no_gender"].append(L.eer_from_scores(sc, lab))
    return float(np.mean(out["no_gender"])), float(np.mean(out["gender"]))

V6144 = VOICE_FEATS["train"]["ecapa6144"]
V192 = VOICE_FEATS["train"]["ecapa192"]
FARC = FACE_FEATS["train"]

runs = {
    "A control (given 4096 + given 192)": (Xf_old, Xv_old),
    "B given face + ECAPA 6144":          (Xf_old, V6144),
    "C given face + our ECAPA 192":       (Xf_old, V192),
    "D ArcFace 512 + given 192":          (FARC,   Xv_old),
    "E ArcFace 512 + ECAPA 6144":         (FARC,   V6144),
}
rows = []
for tag, (a, b) in runs.items():
    t0 = time.time()
    ng, g = evaluate(a, b, tag)
    rows.append(dict(run=tag, face_dim=a.shape[1], voice_dim=b.shape[1],
                     int_ng=round(ng, 2), int_g=round(g, 2),
                     proxy_avg=round((ng + g) / 2, 2),
                     sec=round(time.time() - t0)))
    pd.DataFrame(rows).to_csv("feature_ablation.csv", index=False)
    print(rows[-1], flush=True)

df = pd.DataFrame(rows)
print()
print(df.to_string(index=False))
print("\nControl int_g was 30.42 locally -> CodaBench 30.90.")
print("proxy_avg = mean of internal standard + gender-constrained English EER.")
print("It is only a local proxy; the official overall score averages FOUR EERs, including Bengali unheard.")
print("If B beats A clearly, the 192-d voice really was the bottleneck.")


## 5. Submission from the winning feature pair


In [ ]:
# Build a submission from the feature pair with the best balanced local proxy.
# The official score averages four cells; locally we only have labels to simulate the
# two English protocol types, so use mean(int_ng, int_g) rather than int_g alone.
BEST_RUN = df.sort_values(["proxy_avg", "int_g"]).iloc[0]["run"]
print("best run:", BEST_RUN)
FEATS = {"A control (given 4096 + given 192)": ("old", "old"), "B given face + ECAPA 6144": ("old", "6144"),
         "C given face + our ECAPA 192": ("old", "192"), "D ArcFace 512 + given 192": ("arc", "old"),
         "E ArcFace 512 + ECAPA 6144": ("arc", "6144")}[BEST_RUN]

def feats_for(split):
    f = FACE_FEATS[split] if FEATS[0] == "arc" else None
    v = {"old": None, "6144": VOICE_FEATS[split]["ecapa6144"], "192": VOICE_FEATS[split]["ecapa192"]}[FEATS[1]]
    return f, v

Xf_tr, Xv_tr = feats_for("train")
Xf_tr = Xf_tr if Xf_tr is not None else Xf_old
Xv_tr = Xv_tr if Xv_tr is not None else Xv_old

cfg = dict(hid=512, emb=128, epochs=40, drop=0.3, dedup_clip=False)
prep_all = L.Prep(Xf_tr, Xv_tr, key="final")
aF, aV = L.Aligner(prep_all.f(Xf_tr)), L.Aligner(prep_all.v(Xv_tr))
nets = [L.train_model(Xf_tr, Xv_tr, spk, gmap, vgrp, ALL, cfg, seed=1 + 100 * i)[:2] for i in range(10)]
cca = L.RidgeCCA(k=4, reg=1.0, pca_x=min(128, Xf_tr.shape[1])).fit(Xf_tr, Xv_tr)

devs, scores = {}, {}
for key in L.NAMES:
    split = f"{key[0]}/{key[1]}"
    a_old, b_old, t = L.load_dev(*key)
    devs[key] = (a_old, b_old, t)
    f, v = feats_for(split)
    a = f if f is not None else a_old
    b = v if v is not None else b_old
    embs = [L.embed_one(net, prep, a, b, alF=aF, alV=aV) for net, prep in nets]
    cca_emb = (cca.transform_x(L.center_to(a, Xf_tr.mean(0))), cca.transform_y(L.center_to(b, Xv_tr.mean(0))))
    ii = np.arange(len(t))
    scores[key] = L.fuse_scores(embs, ii, ii, cca=cca_emb, cca_weight=0.25)

L.write_submission("submission_newfeat.zip", scores, devs)
print("submission polarity: HIGHER score = more likely same speaker (official FLAG rule)")
for k, fn in L.NAMES.items():
    d = pd.read_csv(zipfile.ZipFile("submission_newfeat.zip").open(fn), sep=" ", header=None, names=["pid", "s"])
    print(fn, len(d), bool((d.pid.values == devs[k][2].pair_id.values).all()), int(d.s.isna().sum()))
